# 14 – Token Budget Guard & LLM Factory

Two safety layers around LLM usage:
- **`llm_guard`** — daily token budget with Redis tracking, fail-open
- **`llm_factory`** — LiteLLM-based factory with Groq primary + mock fallback
- **`get_structured_llm`** — returns an LLM bound to a Pydantic schema

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
os.environ['ENABLE_MOCK'] = 'true'
os.environ['REDIS_ENABLED'] = 'false'

## 1. Token estimation

In [ ]:
from core.llm_guard import estimate_tokens, DAILY_TOKEN_LIMIT

samples = [
    'What is GRR?',
    'Why did retention drop last month? Please provide a detailed analysis including all relevant metrics.',
    'x' * 1000,
]

print(f'Daily token limit: {DAILY_TOKEN_LIMIT:,}\n')
for s in samples:
    est = estimate_tokens(s)
    print(f'chars={len(s):5}  estimated tokens={est:6}  text={repr(s[:50])}')

## 2. check_and_record_tokens — no Redis (fail-open)

In [ ]:
from core.llm_guard import check_and_record_tokens, get_daily_usage

# Without Redis, always returns True (fail-open)
allowed = check_and_record_tokens(redis_client=None, tokens=50000)
print(f'Allowed (no Redis): {allowed}')

usage = get_daily_usage(redis_client=None)
print(f'Usage stats (no Redis): {usage}')

## 3. check_and_record_tokens — mock Redis, enforce budget

In [ ]:
from unittest.mock import MagicMock
from core.llm_guard import DAILY_TOKEN_LIMIT

# Mock Redis near the limit
mock_redis = MagicMock()
mock_redis.get.return_value = str(DAILY_TOKEN_LIMIT - 1000).encode()  # 1000 remaining

# Small request — should be allowed
allowed = check_and_record_tokens(mock_redis, tokens=500)
print(f'500 tokens (500 remaining): allowed={allowed}')

# Reset mock to near-full
mock_redis.get.return_value = str(DAILY_TOKEN_LIMIT - 100).encode()

# Large request — should be blocked
blocked = check_and_record_tokens(mock_redis, tokens=50000)
print(f'50000 tokens (100 remaining): allowed={blocked}')

## 4. LLM Factory — MockLLM fallback when no API key

In [ ]:
from core.llm_factory import get_llm, get_structured_llm

# Without GROQ_API_KEY, returns _MockLLM
llm = get_llm()
print('LLM type:', type(llm).__name__)

response = llm.invoke('What is the GRR threshold?')
print('Response:', response.content)

## 5. Structured LLM with Pydantic schema

In [ ]:
from graph.intent import IntentClassification

structured = get_structured_llm(schema=IntentClassification)
print('Structured LLM type:', type(structured).__name__)

# With mock LLM, invoke returns a mock message
r = structured.invoke('classify this query: Why did retention drop?')
print('Response:', r.content)

## 6. Config settings — LLM config

In [ ]:
from config.settings import get_config

cfg = get_config()
print('LLM config:')
print('  provider       :', cfg.llm.provider)
print('  model          :', cfg.llm.model)
print('  primary_model  :', cfg.llm.primary_model)
print('  fallback_models:', cfg.llm.fallback_models)
print('  temperature    :', cfg.llm.temperature)
print('  max_tokens     :', cfg.llm.max_tokens)
print('  api_key set    :', bool(cfg.llm.api_key))